# ReFuelEU optimisation — exploring the scenarios

Every run behind the paper, without running anything. This notebook reads only the
dataset in `dataset/` (see its `README.md` for what each file holds and for the units),
so it needs pandas and matplotlib, and ipywidgets for the interactive cell at the end.

1. **The scenarios** — what was run, and with which parameters
2. **Key outputs** — the numbers a reader usually wants, one row per optimisation
3. **Any variable, any scenarios** — one function to plot a variable across runs
4. **Two scenarios side by side**
5. **Interactive** — the same plot, driven by dropdowns

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import dataset as D
import paper_style as style

style.apply()

scenarios = D.scenarios()
variables = pd.read_csv(D.DATA_DIR / "variables.csv", index_col="variable")
UNITS = variables["unit"]

## 1. The scenarios

96 runs in four families: the carbon-budget **ladder** of five cases, each case at the
**central** ReFuelEU-equivalent budget, one-parameter **sensitivities** around `base`, and
fixed-mandate **references**. Parameters are given as shares of the *world* totals; the
model then downscales every run to the EU perimeter.

In [ ]:
scenarios.groupby("family").size().rename("runs").to_frame()

In [ ]:
PARAMETERS = [
    "label",
    "carbon_budget_world_share_pct",
    "biomass_world_share_pct",
    "price_elasticity",
    "ramp_rate_cap",
    "ramp_volume_cap_EJ_per_yr",
    "social_discount_rate",
    "efuel_pathway",
    "feasible",
]
scenarios.loc[scenarios.family.str.startswith("sensitivity"), PARAMETERS]

## 2. Key outputs

One row per optimisation. Surplus loss is the paper's objective in Bn€, discounted to
2020 and measured against 2019 unit economics, so only differences between rows are
policy costs. Infeasible runs (budgets the ramp-up limits cannot reach) are kept but
flagged: their numbers are a best effort, not an optimum.

In [ ]:
def key_outputs(names):
    """The headline numbers of each run, one row per run."""
    rows = {}
    for name in names:
        v, floats = D.load(name)
        rows[name] = {
            "label": scenarios.loc[name, "label"],
            "feasible": scenarios.loc[name, "feasible"],
            "biofuel 2030 (%)": v["generic_biofuel_share_dropin_fuel"].loc[2030],
            "biofuel 2050 (%)": v["generic_biofuel_share_dropin_fuel"].loc[2050],
            "electrofuel 2030 (%)": v["generic_electrofuel_share_dropin_fuel"].loc[2030],
            "electrofuel 2050 (%)": v["generic_electrofuel_share_dropin_fuel"].loc[2050],
            "CO2 2050 (Mt)": v["co2_emissions_including_energy"].loc[2050],
            "cumulative CO2 2020-2050 (Gt)": v["cumulative_co2_emissions"].loc[2050],
            "RPK 2050 (Tpkm)": v["rpk"].loc[2050] / 1e12,
            "airfare 2050 (EUR/pkm)": v["airfare_per_rpk"].loc[2050],
            # fixed_demand has no demand response, hence no surplus: its objective is cost
            "surplus loss (Bn EUR)": v["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9
            if "cumulative_total_surplus_loss_discounted" in v
            else float("nan"),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


optimisations = scenarios.index[scenarios.objective != "none"]
outputs = key_outputs(optimisations)
outputs.round(2)

The same table for one family or case is a filter away, e.g. the `main` ladder:

In [ ]:
outputs.loc[
    [name for name in outputs.index if name.startswith("opt_main_") and "refueleu" not in name]
].round(2)

## 3. Any variable, any scenarios

`plot(variable, names)` draws one variable for a list of runs; `variables` lists what is
available, with units.

In [ ]:
variables

In [ ]:
def plot(variable, names, start=2019, ax=None):
    """One variable over time, one line per run."""
    ax = ax or plt.subplots(figsize=(8, 4.5))[1]
    for name in names:
        series = D.load(name)[0][variable].loc[start:]
        ax.plot(series.index, series, label=scenarios.loc[name, "label"])
    ax.set_title(variables.loc[variable, "description"])
    ax.set_ylabel(UNITS[variable])
    ax.legend(fontsize=8)
    return ax


plot(
    "generic_biofuel_share_dropin_fuel",
    ["opt_main_3_8", "opt_main_3_0", "opt_main_2_4", "refueleu_main"],
);

In [ ]:
plot(
    "co2_emissions_including_energy",
    ["fossil_main", "refueleu_main", "opt_main_refueleu", "opt_pess_refueleu"],
);

## 4. Two scenarios side by side

`compare(a, b)` gives every variable at a few milestone years for both runs, and the
difference. Below: the reference case against the pessimistic technology roadmap, both
at the ReFuelEU-equivalent budget.

In [ ]:
def compare(a, b, years=(2030, 2040, 2050)):
    """Every variable of runs a and b at the given years, and b - a."""
    va, vb = D.load(a)[0], D.load(b)[0]
    common = [v for v in va.columns if v in vb.columns]
    frame = pd.concat(
        {
            a: va.loc[list(years), common].T,
            b: vb.loc[list(years), common].T,
            f"{b} - {a}": (vb - va).loc[list(years), common].T,
        },
        axis=1,
    )
    frame.insert(0, "unit", UNITS.reindex(common))
    return frame


compare("opt_main_refueleu", "opt_pess_refueleu")

## 5. Interactive

Pick a family and a case, a variable, and any number of runs (ctrl/cmd-click). Needs a
live kernel with ipywidgets.

In [ ]:
import ipywidgets as widgets

family = widgets.Dropdown(options=["all", *sorted(scenarios.family.unique())], description="family")
case = widgets.Dropdown(options=["all", *sorted(scenarios.case.unique())], description="case")
variable = widgets.Dropdown(options=list(variables.index), value="rpk", description="variable")
runs = widgets.SelectMultiple(rows=12, description="runs", layout=widgets.Layout(width="60%"))
output = widgets.Output()


def _filter(*_):
    keep = scenarios
    if family.value != "all":
        keep = keep[keep.family == family.value]
    if case.value != "all":
        keep = keep[keep.case == case.value]
    runs.options = [(f"{name} - {row.label}", name) for name, row in keep.iterrows()]


def _draw(*_):
    with output:
        output.clear_output(wait=True)
        if runs.value:
            plot(variable.value, runs.value)
            plt.show()


family.observe(_filter, "value")
case.observe(_filter, "value")
variable.observe(_draw, "value")
runs.observe(_draw, "value")
_filter()
widgets.VBox([widgets.HBox([family, case, variable]), runs, output])